In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia_typed = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_raw = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_categories.head()


1.

In [ ]:
types = pd.concat(
    {
        "raw": data_gdynia_raw.dtypes,
        "typed": data_gdynia_typed.dtypes,
        "is_empty": data_gdynia_typed.isna().all(),
    },
    axis=1,
)

In [ ]:
diff_types = types[(types["raw"] != types["typed"]) & ~types["is_empty"]]
empty = types[types["is_empty"]]


In [ ]:
assert len(diff_types) == 5
assert (empty["raw"] == "float64").all()

2.

In [ ]:
raw_na = data_gdynia_raw["confidence"].where(
    data_gdynia_raw["arch_style"] != "modernizm"
)
typed_na = data_gdynia_typed["confidence"].where(
    data_gdynia_typed["arch_style"] != "modernizm"
)

In [ ]:
print(f"dtype: raw wo Nan: {data_gdynia_raw['confidence'].dtype}")
print(f"dtype: raw with Nan: {raw_na.dtype}")
print(f"dtype: typed wo Nan: {data_gdynia_typed['confidence'].dtype}")
print(f"dtype: typed with Nan: {typed_na.dtype}")
print(f"Nan value raw: {raw_na[raw_na.isna()].head(1)}")
print(f"Nan value typed: {typed_na[typed_na.isna()].head(1)}")

In [ ]:
assert raw_na.isna().sum() == typed_na.isna().sum() == 466
assert raw_na.mean() == typed_na.mean()
assert np.isclose(raw_na.mean(), 2.4574, atol=0.001)
assert raw_na.sum() == typed_na.sum() == 1268

3.

In [ ]:
atrybutes = (
    data_gdynia_typed.loc[
        :, ["osm_id", "addr_street", "source", "number_of_storeys", "date_of_checking"]
    ]
    .agg(["size", "count", "nunique", "hasnans", "is_unique"])
    .T
)
atrybutes

In [ ]:
assert atrybutes.loc["osm_id", "hasnans"] == data_gdynia_typed["osm_id"].hasnans
assert atrybutes.loc["osm_id", "is_unique"] == data_gdynia_typed["osm_id"].is_unique

assert (atrybutes.loc["addr_street"] == [982, 713, 44, True, False]).all()

4.

In [ ]:
data_categories_styles = data_gdynia_typed.assign(
    arch_style=lambda d: d["arch_style"].astype("category")
)


In [ ]:
data_categories_styles["arch_style"].cat.categories

In [ ]:
data_gdynia_typed.loc[
    data_gdynia_typed["addr_street"] == "Świętojańska", "arch_style"
].value_counts()

In [ ]:
data_categories_Sw = data_categories_styles.loc[
    data_categories_styles["addr_street"] == "Świętojańska", "arch_style"
].value_counts()
data_categories_Sw

In [ ]:
assert (data_categories_Sw == [90, 9, 7, 0, 0]).all()

In [ ]:
data_gdynia_long_categories["category"].value_counts()

5.

In [ ]:
sum_vacant = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] == "pustostan"
    ]
    .groupby("addr_street")
    .size()
)
len(sum_vacant)

In [ ]:
sum_functions = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] != "brak parteru"
    ]
    .groupby("addr_street")
    .size()
)
len(sum_functions)

In [ ]:
pd.testing.assert_series_equal(
    sum_vacant.div(sum_functions), sum_vacant / sum_functions
)

In [ ]:
sum_vacant.div(sum_functions, fill_value=0).rename("share_vacant")

6.

In [ ]:
share_vacant = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] != "brak parteru"
    ]
    .assign(is_vacant=lambda d: d["category"] == "pustostan")
    .groupby("addr_street")
    .agg(
        num_premises=("addr_street", "size"),
        num_vacant=("is_vacant", "sum"),
        share_vacant=("is_vacant", "mean"),
    )
)


In [ ]:
pd.testing.assert_series_equal(
    sum_vacant.div(sum_functions, fill_value=0).rename("share_vacant"),
    share_vacant["share_vacant"],
)